In [2]:
from IncrementalRBFSVC import IncrementalRBFSVC
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingRandomSearchCV
from fractions import Fraction

In [3]:
import pandas as pd
df = pd.read_csv('../output/Bach_chordify_roman_5.csv')
display(df.head())
df.drop(['file', 'position'], axis=1, inplace=True)

,lookback_1,lookback_2,lookback_3,lookback_4,lookback_5,label,duration,file,position
0,START,START,START,START,START,iii,0.25,"Bach, Carl Philipp Emanuel, Keyboard Sonata in...",0
1,START,START,START,START,iii,vi,0.25,"Bach, Carl Philipp Emanuel, Keyboard Sonata in...",1
2,START,START,START,iii,vi,vi,0.25,"Bach, Carl Philipp Emanuel, Keyboard Sonata in...",2
3,START,START,iii,vi,vi,v,0.50,"Bach, Carl Philipp Emanuel, Keyboard Sonata in...",3
4,START,iii,vi,vi,v,#i,0.50,"Bach, Carl Philipp Emanuel, Keyboard Sonata in...",4


In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
import pandas as pd
from sklearn.preprocessing import LabelEncoder
# Encode columns in for the use of strings.
lookback_cols = [c for c in df.columns if 'lookback' in c and df[c].dtype == 'object' or 'label' in c]

all_chords = set(df['label'].unique())
for col in lookback_cols:
    all_chords |= set(df[col].dropna().unique())

le_chord = LabelEncoder()
le_chord.fit(sorted(all_chords))

for col in lookback_cols:
    df[col] = le_chord.transform(df[col])


display(df.head())

,lookback_1,lookback_2,lookback_3,lookback_4,lookback_5,label,duration
0,9,9,9,9,9,20,0.25
1,9,9,9,9,20,23,0.25
2,9,9,9,20,23,23,0.25
3,9,9,20,23,23,22,0.50
4,9,20,23,23,22,0,0.50


In [24]:
X_chord = df.drop(columns=['label', 'duration'])
y_chord = df['label']
X_duration = df.drop(columns=['duration'])
df['duration'] = df['duration'].apply(lambda x: Fraction(x).limit_denominator())
from sklearn.preprocessing import LabelEncoder
le_dur = LabelEncoder()
le_dur.fit(sorted(df['duration'].unique()))
df['duration'] = le_dur.transform(df['duration'])
y_duration = df['duration']
print("Training samples:", X_chord.shape[0])
print("Feature dims:",    X_chord.shape[1])
display(X_chord.head())
display(y_duration.head())

Training samples: 200525
Feature dims: 1


,lookback_5
0,9
1,20
2,23
3,23
4,22


0    2
1    2
2    2
3    5
4    5
Name: duration, dtype: int64

In [26]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import learning_curve, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import RBFSampler
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
#Split data into testing and training set
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_chord, y_chord, test_size=0.2, random_state=42)
Xd_train, Xd_test, yd_train, yd_test = train_test_split(X_duration, y_duration, test_size=0.2, random_state=42)

In [27]:
# Train data using the IncrementalRBFSVC method, which approximates an RBF SVC.
clf_chord = IncrementalRBFSVC(gamma='auto', loss='modified_huber', C=1, chunk_size=1000, random_state=42, verbose=True)
clf_duration = IncrementalRBFSVC(gamma='auto', loss='modified_huber', C=1, chunk_size=1000, random_state=42, verbose=True)
clf_chord.fit(Xc_train, yc_train)
clf_duration.fit(Xd_train, yd_train)
print("Test accuracy (chord):", clf_chord.score(Xc_test, yc_test))
print("Test accuracy (duration):", clf_duration.score(Xd_test, yd_test))

Fitting chunks:   0%|          | 0/161 [00:00<?, ?chunk/s]

Fitting chunks:   0%|          | 0/161 [00:00<?, ?chunk/s]

Test accuracy (chord): 0.1896770976187508
Test accuracy (duration): 0.38623613015833436


In [15]:
import numpy as np
import pandas as pd

chord_cols = clf_chord.feature_names_in_
dur_cols   = clf_duration.feature_names_in_

#Add starting point for the generation. For the sake of simplicity, starting point was set to beginning.
init_chord_ctx = list(Xc_test.iloc[0])
init_dur_ctx   = list(Xd_test.iloc[0])[: len(Xd_test.iloc[0]) - len(init_chord_ctx)]

steps      = 128
generated  = []
chord_ctx  = init_chord_ctx.copy()
dur_ctx    = init_dur_ctx.copy()

for step in range(steps):
    # Generate Chord
    Xc_df = pd.DataFrame([chord_ctx], columns=chord_cols)
    proba_c = clf_chord.predict_proba(Xc_df)[0]
    enc_chord = np.random.choice(clf_chord.classes_, p=proba_c)
    # Genderate Duration
    chord_ctx = chord_ctx[1:] + [enc_chord]
    Xd_df = pd.DataFrame([chord_ctx + dur_ctx], columns=dur_cols)
    proba_d = clf_duration.predict_proba(Xd_df)[0]
    next_dur = np.random.choice(clf_duration.classes_, p=proba_d)
    dur_ctx = dur_ctx[1:] + [next_dur]
    
    generated.append({
        'chord':    enc_chord,
        'duration': next_dur
    })

df_gen = pd.DataFrame(generated)
df_gen['chord'] = le_chord.inverse_transform(df_gen['chord'].values)
df_gen['duration'] = le_dur.inverse_transform(df_gen['duration'].values)
df_gen.to_csv('generated_sequence_batched.csv', index=False)
print(f"\nWrote {len(df_gen)} events to generated_sequence_full.csv")


Wrote 128 events to generated_sequence_full.csv
